# 🔥 Fine-tune Qwen 2.5 7B Instruct untuk K3 — QLoRA + Unsloth (Memory Optimized)

**Author:** Jascon Johanest Kembuan — PNJ  
**Target:** Kaggle T4 (15GB) / RTX 3060 (12GB Desktop atau 6GB Mobile)  
**Optimasi:** Memory-efficient buat hindarin CUDA OOM

---

## 📋 Setup Checklist

1. **Kaggle Notebook** → New → Pilih `GPU T4 x2` (Accelerator → GPU T4 x2)
2. Upload dataset: `k3_qwen7b_train.jsonl` + `k3_qwen7b_val.jsonl`
3. Set `Internet ON` di settings notebook
4. **⚠️ WAJIB:** Restart kernel sebelum run (Run → Restart & Clear Output)
5. Run cells secara sequential

## 🛠️ Perubahan dari Versi Original

| Parameter | Sebelum | Sesudah | Alasan |
|---|---|---|---|
| `MAX_SEQ_LENGTH` | 2048 | 1024 | Hemat ~50% activation memory |
| `per_device_train_batch_size` | 4 | 1 | Fix utama OOM |
| `gradient_accumulation_steps` | 4 | 16 | Effective batch tetep 16 |
| `LoRA rank (r)` | 32 | 16 | Hemat ~30% adapter memory |
| `lora_alpha` | 64 | 32 | Tetep 2x rank (best practice) |
| `optim` | `adamw_8bit` | `paged_adamw_8bit` | Bisa offload optimizer state ke CPU RAM |
| `PYTORCH_CUDA_ALLOC_CONF` | — | `expandable_segments:True` | Reduce memory fragmentation |

**Estimated peak VRAM:** ~5.5–7 GB (turun dari ~16–18 GB)  
**Estimated training time:** ~2–4 jam untuk 3 epoch  
**Output:** `qwen2.5-7b-k3-q4_k_m.gguf` (~4.5 GB)


## Cell 1 — Cek Dataset Path

Pastiin dataset ke-upload bener di Kaggle input.


In [ ]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    print(root, files[:5])


## Cell 2 — Imports & Config (Memory Optimized)

**⚠️ PENTING:** `PYTORCH_CUDA_ALLOC_CONF` **HARUS** di-set sebelum `import torch`, kalau enggak gak ngefek.


In [ ]:
# ============== CELL 2: Imports & Config ==============
import os

# ⚠️ HARUS sebelum import torch — biar reduce memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import torch

# Clear sisa memory dari run sebelumnya (kalau ada)
gc.collect()
torch.cuda.empty_cache()

from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments

# =========================
# Dataset Paths
# =========================
TRAIN_FILE = "/kaggle/input/datasets/jasconkembuan/k3-dataset/k3_qwen7b_train.jsonl"
VAL_FILE = "/kaggle/input/datasets/jasconkembuan/k3-dataset/k3_qwen7b_val.jsonl"

# =========================
# Output
# =========================
OUTPUT_DIR = "/kaggle/working/k3_qwen7b_lora"

# =========================
# Model Config — DIPERKETAT
# =========================
# Turunin dari 2048 → 1024 buat hemat memory.
# Cell 5 akan kasih insight token length actual dataset lo,
# kalau P95 nya < 768, turunin lagi MAX_SEQ_LENGTH ke 768.
MAX_SEQ_LENGTH = 1024

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

# =========================
# Cek Dataset Exists
# =========================
print("Train exists:", os.path.exists(TRAIN_FILE))
print("Val exists:", os.path.exists(VAL_FILE))

# =========================
# GPU Info & Free Memory
# =========================
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    total_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 2)
    free_memory = round(torch.cuda.mem_get_info()[0] / 1024**3, 2)
    print(f"GPU: {gpu_stats.name}")
    print(f"VRAM Total: {total_memory} GB")
    print(f"VRAM Free:  {free_memory} GB")
    
    # Sanity check
    if free_memory < 5:
        print("⚠️ WARNING: Free VRAM < 5GB. Restart kernel dulu sebelum lanjut!")
else:
    print("❌ CUDA not available!")


## Cell 3 — Load Model (4-bit Quantized)

Load Qwen 2.5 7B dalam 4-bit (BitsAndBytes). Empty cache dulu biar fresh.


In [ ]:
# ============== CELL 3: Load Model ==============
import gc
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,                    # Auto-detect (bfloat16 on T4, fp16 on RTX 3060)
    load_in_4bit=True,
)

# Cek memory usage after load
allocated = round(torch.cuda.memory_allocated() / 1024**3, 2)
reserved = round(torch.cuda.memory_reserved() / 1024**3, 2)
print(f"✅ Model loaded")
print(f"   Allocated: {allocated} GB")
print(f"   Reserved:  {reserved} GB")


## Cell 4 — Setup LoRA Adapter (rank 16)

LoRA rank diturunin dari 32 → 16. Buat dataset domain-specific kayak K3 (scope narrow), rank 16 udah cukup dan hemat ~30% adapter memory.

**Trainable params estimate:** ~40M (dari ~7.6B total) = ~0.5%


In [ ]:
# ============== CELL 4: Setup LoRA Adapter ==============
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                          # ⬇️ dari 32 → 16 (hemat memory, masih cukup buat K3)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,                 # ⬇️ tetep 2x rank
    lora_dropout=0,                # 0 = best buat QLoRA
    bias="none",
    use_gradient_checkpointing="unsloth",   # ✅ KRUSIAL — jangan diubah
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# Print trainable params
model.print_trainable_parameters()


## Cell 5 — Load & Format Dataset + Token Length Analysis

Bagian tambahan: gue tambahin analisis token length biar lo bisa fine-tune `MAX_SEQ_LENGTH` lebih akurat.


In [ ]:
# ============== CELL 5: Load & Format Dataset ==============
from datasets import load_dataset

# Format chat template
def format_chat(example):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# Load dataset
train_ds = load_dataset("json", data_files=TRAIN_FILE, split="train")
val_ds = load_dataset("json", data_files=VAL_FILE, split="train")

# Apply formatting
train_ds = train_ds.map(format_chat, remove_columns=train_ds.column_names)
val_ds = val_ds.map(format_chat, remove_columns=val_ds.column_names)

# =========================
# Token Length Analysis (TAMBAHAN)
# =========================
print(f"Train: {len(train_ds)} samples")
print(f"Val:   {len(val_ds)} samples")

# Hitung distribusi token length
import numpy as np
lengths = [len(tokenizer.encode(s["text"])) for s in train_ds]
lengths = np.array(lengths)

print("\n📊 Token Length Stats (train set):")
print(f"   Min:   {lengths.min()}")
print(f"   Mean:  {lengths.mean():.0f}")
print(f"   P50:   {np.percentile(lengths, 50):.0f}")
print(f"   P95:   {np.percentile(lengths, 95):.0f}")
print(f"   P99:   {np.percentile(lengths, 99):.0f}")
print(f"   Max:   {lengths.max()}")

# Saran MAX_SEQ_LENGTH
p95 = int(np.percentile(lengths, 95))
print(f"\n💡 Saran: Set MAX_SEQ_LENGTH ke {((p95 // 128) + 1) * 128} (P95 round up ke kelipatan 128)")
print(f"   Current MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")

# Preview sample
print("\n📝 Sample formatted:")
print(train_ds[0]["text"][:500])


## Cell 6 — Training Config (FIX UTAMA)

Ini bagian paling impactful buat fix OOM:

- `per_device_train_batch_size=1` (dari 4) — fix utama
- `gradient_accumulation_steps=16` (dari 4) — effective batch tetep 16
- `optim="paged_adamw_8bit"` — bisa offload state ke CPU RAM kalau VRAM tight
- `dataloader_pin_memory=False` — hemat sedikit memory di laptop


In [ ]:
# ============== CELL 6: Training Config ==============
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # === Batch & Accumulation (FIX UTAMA OOM) ===
    per_device_train_batch_size=1,         # ⬇️ dari 4 → 1
    gradient_accumulation_steps=16,        # ⬆️ dari 4 → 16 (effective batch = 16)
    per_device_eval_batch_size=1,          # ➕ eksplisit, biar eval gak OOM
    
    # === Training Schedule ===
    warmup_steps=20,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    
    # === Precision ===
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    
    # === Optimizer (paged version, bisa offload ke CPU RAM) ===
    optim="paged_adamw_8bit",              # 🔄 dari adamw_8bit
    
    # === Logging & Saving ===
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    
    # === Misc Memory Optimization ===
    gradient_checkpointing=False,          # Unsloth udah handle — jangan double
    dataloader_pin_memory=False,           # Hemat sedikit
    dataloader_num_workers=0,              # Buat laptop, reduce overhead
    
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("✅ Trainer ready")
print(f"   Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Total training steps: ~{(len(train_ds) // 16) * 3}")


## Cell 7 — Start Training 🚀

Final memory check, terus start training. Kalau ada OOM di sini, lihat tips troubleshooting di bawah.


In [ ]:
# ============== CELL 7: Start Training ==============
gc.collect()
torch.cuda.empty_cache()

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)

print(f"GPU: {gpu_stats.name}")
print(f"Max memory: {max_memory} GB")
print(f"Reserved at start: {start_gpu_memory} GB")
print(f"Free now: {round(torch.cuda.mem_get_info()[0] / 1024**3, 2)} GB")
print("\n🚀 Starting training...\n")

trainer_stats = trainer.train()

# Print final memory usage
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\n✅ Training done")
print(f"   Peak memory: {used_memory} GB / {max_memory} GB ({used_memory/max_memory*100:.1f}%)")
print(f"   Time: {trainer_stats.metrics['train_runtime']:.1f} sec")


## Cell 8 — Save LoRA Adapter

In [ ]:
# ============== CELL 8: Save LoRA Adapter ==============
model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
print(f"✅ LoRA adapter saved to {OUTPUT_DIR}/final_adapter")


## Cell 9 — Quick Inference Test

Tes model dengan beberapa pertanyaan K3 buat cek apakah fine-tuning udah masuk.


In [ ]:
# ============== CELL 9: Quick Inference Test ==============
FastLanguageModel.for_inference(model)

test_questions = [
    "Gimana cara pakai APAR yang benar?",
    "Klasifikasi kebakaran kelas B itu apa?",
    "Score LSTM 1 artinya apa?",
    "Kalau ada notif bahaya di Telegram harus apa?",
    "Sensor MQ-7 itu untuk apa?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": "Kamu adalah Asisten K3 untuk sistem deteksi kebakaran IoT. Jawab ringkas, faktual, Bahasa Indonesia baku. Maksimal 3 kalimat kecuali user minta lebih panjang."},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=200,
        use_cache=True,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\n❓ Q: {q}")
    print(f"💬 A: {response.strip()}")


## Cell 10 — Merge & Save as GGUF

Convert LoRA adapter + base model → single GGUF file buat deploy di RTX 3060 lo via `llama-cpp-python` atau Ollama.

- `q4_k_m` = ~4.5 GB, balance bagus size/quality (recommended)
- `q5_k_m` = ~5.5 GB, quality lebih bagus dikit
- `q8_0` = ~8 GB, mendekati original quality


In [ ]:
# ============== CELL 10: Merge & Save as GGUF ==============
model.save_pretrained_gguf(
    "/kaggle/working/k3_qwen7b_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("✅ GGUF saved to /kaggle/working/k3_qwen7b_gguf/")
print("   Download file *.gguf untuk deploy di RTX 3060 lo")


## Cell 11 — Download Helper

List file GGUF yang siap di-download. Di Kaggle, pakai tab `Output` di sidebar buat download via browser.


In [ ]:
# ============== CELL 11: Download Helper ==============
import os
gguf_files = []
for root, dirs, files in os.walk("/kaggle/working/k3_qwen7b_gguf"):
    for f in files:
        if f.endswith(".gguf"):
            full = os.path.join(root, f)
            size_mb = os.path.getsize(full) / (1024 * 1024)
            gguf_files.append((full, size_mb))

print("GGUF files ready to download:")
for fp, size in gguf_files:
    print(f"  {fp} ({size:.1f} MB)")
print("\n💡 Tip: Pakai 'Output' tab di Kaggle buat download via browser, atau kompres dulu kalau > 5GB.")


## 🆘 Troubleshooting OOM

Kalau masih OOM setelah semua optimasi di atas:

### Level 1 — Tweaks Ringan

```python
# Di Cell 2, ganti env var jadi:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

# Di Cell 2, turunin lagi:
MAX_SEQ_LENGTH = 768  # atau 512 kalau dataset lo pendek-pendek
```

### Level 2 — Kurangi Adapter

```python
# Di Cell 4:
r=8                    # turunin lagi rank
lora_alpha=16          # tetep 2x rank

# Kurangi target_modules ke attention only (skip MLP):
target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
```

### Level 3 — Switch Model

Kalau RTX 3060 Mobile 6GB, **jangan paksain 7B**. Switch ke model lebih kecil:

```python
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"   # ~2 GB VRAM
# atau
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit" # ~1 GB VRAM
```

### Level 4 — Strategi Terbaik (Recommended)

**Train di cloud (Kaggle/Colab T4), deploy di laptop:**

1. Training pakai notebook ini di **Kaggle T4** (gratis, 15GB VRAM, 30 jam/minggu)
2. Export ke GGUF (Cell 10)
3. Download GGUF ke laptop RTX 3060
4. Run inference pakai `llama-cpp-python` atau Ollama (cuma butuh ~5-6 GB VRAM buat inference 7B Q4)

```bash
# Di laptop RTX 3060 lo:
pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
```

```python
from llama_cpp import Llama

llm = Llama(
    model_path="./qwen2.5-7b-k3-q4_k_m.gguf",
    n_gpu_layers=-1,        # offload ALL layers ke GPU
    n_ctx=2048,
    verbose=False,
)

output = llm("Q: Gimana cara pakai APAR? A:", max_tokens=200)
print(output["choices"][0]["text"])
```

### Atau Integrasi ke FastAPI Backend Lo

Setelah punya GGUF, lo bisa integrate jadi chatbot di sistem fire detection lo:

```python
# chatbot.py (FastAPI endpoint)
from fastapi import APIRouter
from pydantic import BaseModel
from llama_cpp import Llama

router = APIRouter()

llm = Llama(
    model_path="./models/qwen2.5-7b-k3-q4_k_m.gguf",
    n_gpu_layers=-1,
    n_ctx=2048,
)

class ChatRequest(BaseModel):
    question: str
    sensor_context: dict | None = None  # buat injection data sensor real-time

@router.post("/chat")
async def chat(req: ChatRequest):
    system = "Kamu adalah Asisten K3 untuk sistem deteksi kebakaran IoT..."
    if req.sensor_context:
        system += f"\nData sensor terkini: {req.sensor_context}"
    
    prompt = f"<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\n{req.question}<|im_end|>\n<|im_start|>assistant\n"
    
    output = llm(prompt, max_tokens=300, temperature=0.3, stop=["<|im_end|>"])
    return {"answer": output["choices"][0]["text"].strip()}
```

Ini cocok banget buat RAG-style integration sama sensor data lo. 🔥
